In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

## Loading the Trained Model and Preprocessing Objects

- `load_model('model.h5')` loads the trained ANN model.
- `pickle.load()` loads the saved preprocessing objects from `.pkl` files.
- These objects are reused so that new input data is processed exactly like the training data.

### Objects Loaded

| File | Loaded Object | Purpose |
|------|---------------|---------|
| `model.h5` | Trained ANN Model | Makes predictions on new data. |
| `onehot_encoder_geo.pkl` | OneHotEncoder | Encodes Geography into the same one-hot columns as training. |
| `label_encoder_gender.pkl` | LabelEncoder | Encodes Gender using the same mapping as training. |
| `scaler.pkl` | StandardScaler | Scales input data using the same mean and standard deviation learned during training. |

**Summary:**  
Before making predictions, load the trained model and all saved preprocessing objects to ensure **consistent preprocessing and accurate predictions**.

In [2]:
### Load the trained model, scaler pickle,onehot
model=load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [15]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}


## Preprocessing User Input for Prediction

Before making a prediction, the new user input must be preprocessed exactly the same way as the training data. This ensures the model receives data in the format it was trained on.

### Step 1: Convert Input to DataFrame
- Convert the user's input dictionary into a Pandas DataFrame.
- This makes the input compatible with Pandas and Scikit-learn preprocessing methods.

### Step 2: Label Encode `Gender`
- Use the saved `LabelEncoder` to convert `Male/Female` into numerical values.
- The saved encoder ensures the same mapping used during training is applied during prediction.

### Step 3: One-Hot Encode `Geography`
- Use the saved `OneHotEncoder` to convert the `Geography` column into multiple binary columns.
- Example:
  - France → `[1,0,0]`
  - Germany → `[0,1,0]`
  - Spain → `[0,0,1]`
- The encoder also preserves the exact column order used during training.

### Step 4: Merge Encoded Features
- Drop the original `Geography` column.
- Concatenate the one-hot encoded geography columns with the remaining features.
- The final DataFrame now has the same feature structure as the training dataset.

### Step 5: Scale the Input
- Use the saved `StandardScaler` to normalize the input features.
- The scaler uses the **same mean and standard deviation learned from the training data**.
- This ensures consistent feature scaling and prevents data mismatch.

### Final Output
- The processed data (`input_scaled`) is now in the same format as the training data.
- It can be directly passed to the trained ANN model for prediction.
```


In [26]:
# One-hot encode the 'Geography' input using the saved OneHotEncoder.
# Example: Germany -> [0, 1, 0]

geo_encoded = label_encoder_geo.transform(
    [[input_data['Geography']]]
).toarray()

# Convert the encoded array into a DataFrame.
# The column names are the same as those used during training.
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=label_encoder_geo.get_feature_names_out(['Geography'])
)

# Display the one-hot encoded Geography DataFrame.
geo_encoded_df

c:\Users\Omkar\Desktop\Python\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [29]:
input_df=pd.DataFrame([input_data])
# Convert the user input dictionary into a Pandas DataFrame.
# Wrapping input_data inside [] creates a DataFrame with a single row.

input_df = pd.DataFrame([input_data])

# Display the input DataFrame.
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [30]:
# Encode the 'Gender' column using the saved LabelEncoder.
# Example: Male -> 1, Female -> 0
# This ensures the same encoding used during training is applied during prediction.

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

# Display the updated DataFrame.
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [31]:
# Drop the original 'Geography' column because it has been one-hot encoded.
# Concatenate the encoded geography columns with the remaining features.
# The final DataFrame now has the same structure as the training data.

input_df = pd.concat(
    [input_df.drop("Geography", axis=1), geo_encoded_df],
    axis=1
)

# Display the updated DataFrame.
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [33]:
# Scale the input data using the saved StandardScaler.
# The scaler uses the same mean and standard deviation
# learned from the training data.
# This ensures the new input is scaled exactly like the training data.

input_scaled = scaler.transform(input_df)

# Display the scaled input data.
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [36]:
## PRedict churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


array([[0.02968762]], dtype=float32)

In [37]:
prediction_proba = prediction[0][0]

In [24]:
prediction_proba

np.float32(0.029687623)

In [25]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
